# ReformLab Quickstart — Carbon Tax Impact in Under 15 Minutes

**What is ReformLab?** ReformLab is an OpenFisca-first platform for analyzing environmental policy impacts on households. It handles data, multi-year projections, indicators, and reproducibility — so you can focus on **policy design**, not infrastructure.

**What you'll learn:**
1. Define a policy using typed objects (`CarbonTaxParameters`)
2. Load a population dataset
3. Run a carbon tax scenario on synthetic French household data
4. See distributional impacts by income decile
5. Modify the policy and compare results
6. Understand how reproducibility is tracked
7. Export results for external analysis

**Prerequisites:** Just `pip install reformlab` — no data downloads, no API keys, no configuration.

**Time:** 10-15 minutes (reading + execution)

---
## 1. Define a Policy

Every ReformLab analysis starts with a **policy object**. Instead of passing raw dictionaries, you instantiate a typed dataclass that carries all the parameters your policy needs.

ReformLab ships four policy types, all inheriting from `PolicyParameters`:

```mermaid
classDiagram
    class PolicyParameters {
        +rate_schedule: dict[int, float]
        +exemptions: tuple
        +thresholds: tuple
        +covered_categories: tuple
    }
    class CarbonTaxParameters {
        +redistribution_type: str
        +income_weights: dict
    }
    class SubsidyParameters {
        +eligible_categories: tuple
        +income_caps: dict
    }
    class RebateParameters {
        +rebate_type: str
        +income_weights: dict
    }
    class FeebateParameters {
        +pivot_point: float
        +fee_rate: float
        +rebate_rate: float
    }
    PolicyParameters <|-- CarbonTaxParameters
    PolicyParameters <|-- SubsidyParameters
    PolicyParameters <|-- RebateParameters
    PolicyParameters <|-- FeebateParameters
```

You can use any of these built-in types, or **subclass `PolicyParameters`** to create your own (demonstrated in the advanced notebook). Let's start with a carbon tax.

In [ ]:
# Import the public API and typed policy objects
from pathlib import Path

from reformlab import load_population, run_scenario, show
from reformlab.templates.schema import (
    BaselineScenario,
    CarbonTaxParameters,
    PolicyParameters,
    PolicyType,
    YearSchedule,
)

print("ReformLab is installed and ready to use!")

### Define a carbon tax policy

France's carbon tax rate in 2024 is €44/tCO2. Let's define this as a typed policy object. Every field is visible and self-documenting — no hidden configuration.

In [ ]:
# 1. Define the typed policy
policy = CarbonTaxParameters(
    rate_schedule={2025: 44.0},       # €44/tCO2 in 2025
    redistribution_type="",           # No redistribution (pure tax)
)

# Inspect all fields on the policy object
print("=== Carbon Tax Policy ===")
print(f"Type:               {type(policy).__name__}")
print(f"Rate schedule:      {dict(policy.rate_schedule)}")
print(f"Redistribution:     {'None' if not policy.redistribution_type else policy.redistribution_type}")
print(f"Exemptions:         {policy.exemptions}")
print(f"Thresholds:         {policy.thresholds}")
print(f"Covered categories: {policy.covered_categories}")
print(f"\nThis is a frozen dataclass — immutable after creation.")

### Wrap the policy in a scenario

A **scenario** bundles a policy with a name, year schedule, and description. This is how you tell ReformLab *what* to simulate.

In [ ]:
# 2. Wrap in a scenario
scenario = BaselineScenario(
    name="france-carbon-tax-2025",
    policy_type=PolicyType.CARBON_TAX,
    year_schedule=YearSchedule(start_year=2025, end_year=2025),
    policy=policy,
    description="France's carbon tax at current rate",
)

print(f"Scenario: {scenario.name}")
print(f"Policy type: {scenario.policy_type.value}")
print(f"Years: {scenario.year_schedule.start_year}-{scenario.year_schedule.end_year}")
print(f"Policy type: {type(scenario.policy).__name__}")

---
## 2. Load Population Data

Every ReformLab simulation runs on **population data**: a table of households with income, carbon emissions, and other attributes. In production you'd use real survey data (e.g., INSEE or EU-SILC). For this quickstart, we ship a pre-generated CSV with 100 synthetic French households.

Let's load it and inspect the data our policy will apply to.

In [ ]:
# Load the demo population CSV (shipped with the repo)
# Resolve path relative to this notebook's location (works in both local and CI)
_NB_DIR = Path(__file__).parent if "__file__" in dir() else Path(".")
POPULATION_PATH = (_NB_DIR / "../data/populations/demo-quickstart-100.csv").resolve()

population = load_population(POPULATION_PATH)
pop_table = population.tables["default"]
print(f"Population: {pop_table.num_rows} households")
print(f"Columns: {pop_table.column_names}")
print()
show(pop_table, n=5)

### Ready to run

With a typed scenario and population data loaded, you can pass them directly to `run_scenario()`. No manual bridging to dict-based configs needed — the typed policy flows all the way through to the computation engine.

In [ ]:
# Summary of what we're about to run
print(f"Scenario:    {scenario.name}")
print(f"Policy:      {type(scenario.policy).__name__} — €{policy.rate_schedule[2025]}/tCO2")
print(f"Population:  {pop_table.num_rows} households")
print(f"Years:       {scenario.year_schedule.start_year}-{scenario.year_schedule.end_year}")

---
## 3. Run the Simulation

ReformLab separates **what** you want to simulate from **how** it gets computed:

- **Policy** (what): the `CarbonTaxParameters` object you defined above — rates, exemptions, redistribution rules
- **Adapter** (how): the computation engine that applies the policy to population data

This separation is the **adapter pattern**. The same policy can be computed by different engines: an OpenFisca adapter for production (full French tax-benefit microsimulation), or a custom adapter you define yourself.

An adapter is any class with two methods:
- `version() → str` — identifies the engine
- `compute(population, policy, period) → ComputationResult` — runs the computation

Let's define one right here so you can see exactly what happens inside.

In [ ]:
import pyarrow as pa

# ComputationResult is the standard return type for all adapters.
# It wraps the output PyArrow table together with metadata (adapter version,
# simulation period, timing info) so the orchestrator can track provenance
# and build the run manifest automatically.
from reformlab.computation.types import ComputationResult, PolicyConfig, PopulationData


class SimpleCarbonTaxAdapter:
    """A minimal adapter that applies a flat carbon tax.

    Input population must have columns:
        - household_id (int)       — unique household identifier
        - income (float)           — annual household income in €
        - carbon_emissions (float) — annual CO₂ emissions in tonnes

    Output table will have columns:
        - household_id (int)
        - year (int)               — the simulation period
        - income (float)           — unchanged from input
        - carbon_tax (float)       — tax owed for this household
        - disposable_income (float) — income after tax
    """

    def version(self) -> str:
        return "simple-carbon-tax-v1"

    def compute(
        self,
        population: PopulationData,
        policy: PolicyConfig,
        period: int,
    ) -> ComputationResult:
        # 1. Read the tax rate from the typed policy for this year
        rate = policy.policy.rate_schedule.get(period, 0.0)

        # 2. Read population columns
        table = population.tables["default"]
        household_ids = table.column("household_id")
        incomes = table.column("income").to_pylist()
        emissions = table.column("carbon_emissions").to_pylist()
        n = len(incomes)

        # 3. Apply the formula:
        #      carbon_tax = emissions × (rate / 44.0)
        #      disposable_income = income − carbon_tax
        #
        #    The 44.0 divisor converts €/tCO₂ to a per-unit multiplier
        #    (France's 2024 baseline rate is €44/tCO₂).
        rate_multiplier = rate / 44.0
        carbon_taxes = [max(0.0, e * rate_multiplier) for e in emissions]
        disposable = [incomes[i] - carbon_taxes[i] for i in range(n)]

        # 4. Return a PyArrow table wrapped in ComputationResult
        output = pa.table({
            "household_id": household_ids,
            "year": pa.array([period] * n, type=pa.int64()),
            "income": pa.array(incomes, type=pa.float64()),
            "carbon_tax": pa.array(carbon_taxes, type=pa.float64()),
            "disposable_income": pa.array(disposable, type=pa.float64()),
        })

        return ComputationResult(
            output_fields=output,
            adapter_version=self.version(),
            period=period,
            metadata={"rate": rate},
        )


adapter = SimpleCarbonTaxAdapter()
print(f"Adapter: {adapter.version()}")
print(f"Rate for 2025 will come from the policy: €{policy.rate_schedule[2025]}/tCO₂")

In [ ]:
# Run the simulation — typed scenario goes directly to run_scenario()
print("Running simulation...")
result = run_scenario(scenario, adapter=adapter, population=population, seed=42)
print(result)

In [ ]:
# Inspect the results panel
print(f"Panel shape: {result.panel_output.table.num_rows} rows × {result.panel_output.table.num_columns} columns")
print(f"Columns: {result.panel_output.table.column_names}")
print()
show(result.panel_output.table)

---
## 4. Distributional Analysis — Who Pays the Most?

Carbon taxes are regressive if lower-income households pay a higher share of their income. Let's compute distributional indicators by income decile to see the impact pattern of the policy we defined in Section 1.

ReformLab's indicator system automatically assigns households to income deciles and computes mean, median, sum, min, max for each field.

In [ ]:
# Compute distributional indicators
indicators = result.indicators("distributional")

print(f"Computed {len(indicators.indicators)} indicator records")
print(f"Excluded households (no income): {indicators.excluded_count}")
print(f"Warnings: {indicators.warnings if indicators.warnings else 'None'}")

In [ ]:
import matplotlib.pyplot as plt

# Plot carbon tax burden by income decile
fig, ax = indicators.plot_deciles("carbon_tax")
plt.show()

print("\nInterpretation:")
print("- If bars increase from left to right: tax burden rises with income (progressive in absolute terms)")
print("- If bars are roughly flat or decrease: tax is regressive (lower-income households pay similar/more)")
print("- To see burden as % of income, compute carbon_tax / income ratios (advanced notebook)")

---
## 5. Modify the Policy and Compare

Let's see how the distributional impact changes if we **increase the carbon tax rate** from €44/tCO2 to €100/tCO2. Notice we define a new typed policy object — not a raw dict.

In [ ]:
# Define a new policy with a higher rate
policy_high = CarbonTaxParameters(
    rate_schedule={2025: 100.0},      # €100/tCO2 — more than double
    redistribution_type="",
)

# Wrap in scenario — same adapter, different policy
scenario_high = BaselineScenario(
    name="france-carbon-tax-2025-high",
    policy_type=PolicyType.CARBON_TAX,
    year_schedule=YearSchedule(start_year=2025, end_year=2025),
    policy=policy_high,
    description="France's carbon tax at higher rate",
)

# Reuse the same adapter — it reads the rate from the policy, not from its constructor
print(f"Running with carbon tax rate: €{policy_high.rate_schedule[2025]}/tCO2...")
result_high = run_scenario(scenario_high, adapter=adapter, population=population, seed=42)
print(result_high)

In [ ]:
# Side-by-side comparison using the built-in plot_comparison method
fig, ax = result.plot_comparison(result_high, "carbon_tax", reform_label=f"Reform (€{policy_high.rate_schedule[2025]}/tCO2)")
plt.show()

print("\nComparison shows:")
print("- Reform rate increases burden across all deciles")
print("- Pattern of regressivity (or progressivity) remains similar")
print("- For revenue-neutral reforms (tax + rebate), see advanced notebook")

---
## 6. Reproducibility — Understanding the Run Manifest

Every ReformLab simulation produces a **run manifest** — a complete record of parameters, seeds, data hashes, and versions. This makes results **reproducible**: anyone can verify they're using the same inputs and code.

In [ ]:
# Inspect the manifest
manifest = result.manifest

print("=== Run Manifest ===")
print(f"Manifest ID:      {manifest.manifest_id}")
print(f"Created at:       {manifest.created_at}")
print(f"Engine version:   {manifest.engine_version}")
print(f"Adapter version:  {manifest.adapter_version}")
print(f"Scenario version: {manifest.scenario_version}")
print(f"Policy:           {manifest.policy}")
print(f"Seeds:            {manifest.seeds}")
print(f"Warnings:         {manifest.warnings if manifest.warnings else 'None'}")
print(f"Step pipeline:    {manifest.step_pipeline}")

In [ ]:
# Export the manifest as JSON
import json
import tempfile

manifest_path = result.export_manifest(Path(tempfile.mkdtemp()) / "manifest.json")
print(f"Manifest exported to: {manifest_path}")
print(f"\nManifest contents:")
print(json.dumps(json.loads(manifest_path.read_text()), indent=2))

---
## 7. Export and Next Steps

### Export results for external analysis

ReformLab provides built-in export functionality to save simulation outputs and indicator results in standard formats (CSV/Parquet). All exports include provenance metadata for traceability.

In [ ]:
# Export simulation panel output to CSV and Parquet
export_dir = Path(tempfile.mkdtemp())

csv_path = result.export_csv(export_dir / "simulation_output.csv")
print(f"Panel exported to CSV: {csv_path}")
print(f"  Size: {csv_path.stat().st_size:,} bytes")

parquet_path = result.export_parquet(export_dir / "simulation_output.parquet")
print(f"Panel exported to Parquet: {parquet_path}")
print(f"  Size: {parquet_path.stat().st_size:,} bytes")

print(f"\nExported files contain:")
print(f"  - {result.panel_output.table.num_rows} rows (households × years)")
print(f"  - {result.panel_output.table.num_columns} columns")
print(f"  - Fields: {', '.join(result.panel_output.table.column_names)}")

In [ ]:
# Export indicator results
indicators_csv = indicators.export_csv(export_dir / "indicators_distributional.csv")
print(f"Indicators exported to CSV: {indicators_csv}")
print(f"  Size: {indicators_csv.stat().st_size:,} bytes")

# Verify round-trip: reload and check schema
reloaded = pa_csv.read_csv(indicators_csv)
print(f"\nReloaded CSV verification:")
print(f"  Rows: {reloaded.num_rows}")
print(f"  Columns: {reloaded.column_names}")
print(f"  Sample data:")
show(reloaded, n=5)

### Next Steps

You've just run a complete carbon tax analysis in under 15 minutes! Here's what to explore next:

**Advanced Notebook:**
- Multi-year projections (2025-2034) with escalating rates
- Vintage tracking (how household vehicle fleets evolve)
- Baseline vs. reform comparison with distributional and fiscal indicators
- **Build your own policy type** — subclass `PolicyParameters` to create a custom feebate
- Lineage and reproducibility across scenarios

**Built-in Policy Types:**
- `CarbonTaxParameters` — carbon tax with optional redistribution
- `SubsidyParameters` — subsidies with eligibility criteria and income caps
- `RebateParameters` — rebates with progressive or lump-sum distribution
- `FeebateParameters` — feebate with pivot point and fee/rebate rates

In the advanced notebook, you'll learn to subclass `PolicyParameters` to create your own policy type.

---

**Questions or feedback?** See the project documentation or open an issue on GitHub.